In [18]:
import pandas as pd
import re

archivo = "dataset_fsi.csv"
raw = pd.read_csv(archivo, low_memory=False, encoding="utf-8-sig")
raw.columns = raw.columns.astype(str).str.strip()

trimestres = sorted(
    c for c in raw.columns
    if re.fullmatch(r"\d{4}-Q[1-4]", c)
)

print("Cantidad de columnas trimestrales:", len(trimestres))
print("Primeras:", trimestres[:8])
print("Últimas:", trimestres[-8:])
print("\nFrecuencias:")
print(raw["FREQUENCY"].value_counts(dropna=False).head(10))

Cantidad de columnas trimestrales: 126
Primeras: ['1995-Q1', '1995-Q2', '1995-Q3', '1995-Q4', '1996-Q1', '1996-Q2', '1996-Q3', '1996-Q4']
Últimas: ['2024-Q3', '2024-Q4', '2025-Q1', '2025-Q2', '2025-Q3', '2025-Q4', '2026-Q1', '2026-Q2']

Frecuencias:
FREQUENCY
Quarterly    14967
Annual       14887
Monthly       3473
Name: count, dtype: int64


In [19]:
q = raw[
    raw["FREQUENCY"].astype(str).str.strip().str.upper()
       .isin(["Q", "QUARTERLY"])
].copy()

catalogo = q[
    ["SECTOR", "INDICATOR", "UNIT", "SERIES_NAME"]
].drop_duplicates().copy()

texto = (
    catalogo["INDICATOR"].fillna("").astype(str) + " " +
    catalogo["SERIES_NAME"].fillna("").astype(str)
)

patron = r"non.?performing loans|risk.weighted assets|return on assets|liquid assets to total assets"

display(catalogo[texto.str.contains(patron, case=False, regex=True)].head(100))

,SECTOR,INDICATOR,UNIT,SERIES_NAME
4,Deposit takers,Nonperforming loans net of provisions to capit...,US dollar,Nonperforming loans net of provisions to capit...
5,Deposit takers,"Nonperforming loans to total gross loans, Tota...",Domestic currency,"Nonperforming loans to total gross loans, Tota..."
16,Deposit takers,"Liquid assets to total assets, Liquid assets t...",Percent,"Liquid assets to total assets, Liquid assets t..."
29,Deposit takers,"Return on assets, Net income before taxes, Dom...",Domestic currency,"Return on assets, Net income before taxes, Dep..."
71,Deposit takers,Nonperforming loans net of provisions to capit...,Domestic currency,Nonperforming loans net of provisions to capit...
...,...,...,...,...
15197,Life insurance corporations,"Return on assets (Life insurance only), Net in...",Euro,"Return on assets (Life insurance only), Net in..."
16519,Pension funds,"Return on assets, Annualized net income before...",Euro,"Return on assets, Annualized net income before..."
16533,Pension funds,"Return on assets, Average total assets, Euro",Euro,"Return on assets, Average total assets, Pensio..."
26953,Life insurance corporations,"Return on assets (Life insurance only), Net in...",US dollar,"Return on assets (Life insurance only), Net in..."


In [21]:
SECTOR_ELEGIDO = "Deposit takers"

INDICADORES = {
    "morosidad": "Nonperforming loans to total gross loans, (Core FSI), Percent",
    "capital": "Regulatory capital to risk-weighted assets, (Core FSI), Percent",
    "roa": "Return on assets, (Core FSI), Percent",
    "liquidez": "Liquid assets to total assets, Liquid assets to total assets, Percent",
}

filtrado = q[
    q["SECTOR"].eq(SECTOR_ELEGIDO) &
    q["INDICATOR"].isin(INDICADORES.values())
].copy()

print("Series seleccionadas:", len(filtrado))
display(
    filtrado[
        ["COUNTRY", "SERIES_CODE", "SECTOR", "INDICATOR", "UNIT", "SERIES_NAME"]
    ].head(12)
)

# Convierte todas las columnas trimestrales en filas.
largo = filtrado.melt(
    id_vars=[
        "COUNTRY", "SERIES_CODE", "SECTOR", "INDICATOR", "UNIT",
        "FSI", "FSI_STO", "FSI_COMPONENT", "COUNTERPARTY_SECTOR"
    ],
    value_vars=trimestres,
    var_name="TRIMESTRE",
    value_name="valor"
)

largo["valor"] = pd.to_numeric(largo["valor"], errors="coerce")
largo = largo.dropna(subset=["valor"]).copy()
largo["variable"] = largo["INDICATOR"].map(
    {codigo: nombre for nombre, codigo in INDICADORES.items()}
)
largo["trimestre"] = pd.PeriodIndex(
    largo["TRIMESTRE"].str.replace("-", "", regex=False),
    freq="Q"
)

# Impide mezclar dos versiones de un indicador para el mismo país y trimestre.
claves = ["COUNTRY", "trimestre", "variable"]
duplicados = largo[largo.duplicated(claves, keep=False)]

if not duplicados.empty:
    display(duplicados.head(25))
    raise ValueError(
        "Hay series duplicadas. Revisá UNIT, FSI, FSI_STO y "
        "FSI_COMPONENT antes de continuar."
    )

panel = largo.pivot(
    index=["COUNTRY", "trimestre"],
    columns="variable",
    values="valor"
).reset_index()

faltantes = set(INDICADORES) - set(panel.columns)
if faltantes:
    raise ValueError(f"No se encontraron estos indicadores: {faltantes}")

# Para cada trimestre t, busca la morosidad de exactamente t+4.
futuro = panel[["COUNTRY", "trimestre", "morosidad"]].dropna().copy()
futuro["trimestre"] = futuro["trimestre"] - 4
futuro = futuro.rename(columns={"morosidad": "morosidad_t4"})
panel = panel.merge(futuro, on=["COUNTRY", "trimestre"], how="left")

panel["completo"] = panel[list(INDICADORES)].notna().all(axis=1)
panel["util"] = panel["completo"] & panel["morosidad_t4"].notna()

def racha_maxima(periodos):
    numeros = sorted({p.ordinal for p in periodos})
    mejor = actual = 0
    anterior = None
    for numero in numeros:
        actual = actual + 1 if anterior is not None and numero == anterior + 1 else 1
        mejor = max(mejor, actual)
        anterior = numero
    return mejor

resumen = panel.groupby("COUNTRY").agg(
    trimestres_completos=("completo", "sum"),
    ejemplos_utiles=("util", "sum")
)
resumen["racha_util"] = (
    panel.loc[panel["util"]]
    .groupby("COUNTRY")["trimestre"]
    .agg(racha_maxima)
    .reindex(resumen.index, fill_value=0)
)

display(resumen.sort_values("ejemplos_utiles", ascending=False).head(50))

print(
    "Países con ≥24 ejemplos útiles y ≥16 trimestres seguidos:",
    ((resumen["ejemplos_utiles"] >= 24) &
     (resumen["racha_util"] >= 16)).sum()
)

print("Países utilizables por trimestre reciente:")
display(
    panel.loc[panel["util"]]
    .groupby("trimestre")["COUNTRY"].nunique()
    .tail(12)
)

Series seleccionadas: 623


,COUNTRY,SERIES_CODE,SECTOR,INDICATOR,UNIT,SERIES_NAME
16,"Ethiopia, The Federal Democratic Republic of",ETH.S12CFSI.FSI283_LIQATTA_PT.Q,Deposit takers,"Liquid assets to total assets, Liquid assets t...",Percent,"Liquid assets to total assets, Liquid assets t..."
99,Switzerland,CHE.S12CFSI.FSI283_LIQATTA_PT.Q,Deposit takers,"Liquid assets to total assets, Liquid assets t...",Percent,"Liquid assets to total assets, Liquid assets t..."
163,"Ethiopia, The Federal Democratic Republic of",ETH.S12CFSI.FSI688_CFSI_PT.Q,Deposit takers,"Regulatory capital to risk-weighted assets, (C...",Percent,"Regulatory capital to risk-weighted assets, (C..."
246,"Comoros, Union of the",COM.S12CFSI.FSI688_CFSI_PT.Q,Deposit takers,"Regulatory capital to risk-weighted assets, (C...",Percent,"Regulatory capital to risk-weighted assets, (C..."
265,Russian Federation,RUS.S12CFSI.ROA_CFSI_PT.Q,Deposit takers,"Return on assets, (Core FSI), Percent",Percent,"Return on assets, (Core FSI), Deposit takers, ..."
298,"Comoros, Union of the",COM.S12CFSI.ROA_CFSI_PT.Q,Deposit takers,"Return on assets, (Core FSI), Percent",Percent,"Return on assets, (Core FSI), Deposit takers, ..."
315,"Comoros, Union of the",COM.S12CFSI.FSI283_LIQATTA_PT.Q,Deposit takers,"Liquid assets to total assets, Liquid assets t...",Percent,"Liquid assets to total assets, Liquid assets t..."
548,"Gambia, The",GMB.S12CFSI.ROA_CFSI_PT.Q,Deposit takers,"Return on assets, (Core FSI), Percent",Percent,"Return on assets, (Core FSI), Deposit takers, ..."
558,"Gambia, The",GMB.S12CFSI.AQ12_CFSI_PT.Q,Deposit takers,"Nonperforming loans to total gross loans, (Cor...",Percent,"Nonperforming loans to total gross loans, (Cor..."
593,Singapore,SGP.S12CFSI.FSI688_CFSI_PT.Q,Deposit takers,"Regulatory capital to risk-weighted assets, (C...",Percent,"Regulatory capital to risk-weighted assets, (C..."


,trimestres_completos,ejemplos_utiles,racha_util
COUNTRY,,,
Chile,100,96,96
Ecuador,94,90,90
Bosnia and Herzegovina,91,87,87
Brazil,86,82,82
Canada,86,82,82
Paraguay,86,82,82
Malta,86,82,82
Colombia,86,82,82
Panama,86,82,82


Países con ≥24 ejemplos útiles y ≥16 trimestres seguidos: 129
Países utilizables por trimestre reciente:


trimestre
2022Q3    129
2022Q4    138
2023Q1    127
2023Q2    136
2023Q3    126
2023Q4    135
2024Q1    122
2024Q2    129
2024Q3    114
2024Q4    116
2025Q1     85
2025Q2     53
Freq: Q-DEC, Name: COUNTRY, dtype: int64

In [ ]:
print("Filas que quedaron después de los filtros:", len(q))
print("Columnas del panel:", panel.columns.tolist())
print("Indicadores configurados:", INDICADORES)

if len(q) > 0:
    print("\nIndicadores que quedaron en q:")
    print(
        q[["FREQUENCY", "SECTOR", "INDICATOR", "SERIES_NAME"]]
        .drop_duplicates()
        .head(40)
        .to_string(index=False)
    )

Filas que quedaron después de los filtros: 14967
Columnas del panel: ['COUNTRY', 'trimestre']
Indicadores configurados: {'morosidad': 'PEGAR_INDICATOR_MOROSIDAD', 'capital': 'PEGAR_INDICATOR_CAPITAL', 'roa': 'PEGAR_INDICATOR_RENTABILIDAD', 'liquidez': 'PEGAR_INDICATOR_LIQUIDEZ'}

Indicadores que quedaron en q:
FREQUENCY                       SECTOR                                                                                                                          INDICATOR                                                                                                                                        SERIES_NAME
Quarterly               Deposit takers                                                                                               Return on equity, Capital, US dollar                                                                                               Return on equity, Capital, Deposit takers, US dollar
Quarterly Other financial corporations                

In [ ]:
prefijos = {
    "morosidad": "Nonperforming loans to total gross loans,",
    "capital": "Regulatory capital to risk-weighted assets,",
    "roa": "Return on assets,",
    "liquidez": "Liquid assets to total assets,"
}

INDICADORES = {}

for nombre, prefijo in prefijos.items():
    opciones = sorted(
        q.loc[
            q["SECTOR"].eq("Deposit takers")
            & q["INDICATOR"].astype(str).str.startswith(prefijo)
            & q["INDICATOR"].astype(str).str.strip().str.endswith(", Percent"),
            "INDICATOR"
        ].dropna().unique()
    )

    # Si hay una versión identificada como indicador principal, priorizarla.
    principales = [x for x in opciones if "(Core FSI)" in x]
    if len(principales) == 1:
        opciones = principales

    print(f"\n{nombre}: {opciones}")

    if len(opciones) == 1:
        INDICADORES[nombre] = opciones[0]

print("\nIndicadores identificados:", len(INDICADORES), "de 4")


morosidad: ['Nonperforming loans to total gross loans, (Core FSI), Percent']

capital: ['Regulatory capital to risk-weighted assets, (Core FSI), Percent']

roa: ['Return on assets, (Core FSI), Percent']

liquidez: ['Liquid assets to total assets, Liquid assets to total assets, Percent']

Indicadores identificados: 4 de 4
